# Step 2-4. 내 팀의 라우팅을 증류하기
**Distillation(증류)** 은 큰 모델(스승)의 답을 작은 모델(제자)이 따라 하도록 가르치는 방법입니다. **Step 2-3의 팀**에서 지휘자가 하는 "전문가 고르기"를 작은 모델에게 가르칩니다. (한 단계씩 뜯어 보려면 Step 2-5, 2-6)

사진을 보는 일과 분석은 큰 모델에 그대로 두고, **답의 형태가 정해진 일(누구에게 보낼지)만** 작은 모델로 옮깁니다.

## 순서
1. 질문을 만들고 스승(큰 모델)에게 답을 달게 한다 (2\~3분)
2. 학습 전의 제자에게 시험을 보게 한다
3. 학습시킨다 (5분쯤)
4. 학습 후 같은 시험을 보게 하고, 지휘자 자리에 넣는다

In [ ]:
# 필요한 패키지를 설치합니다. 처음에는 1~2분 걸립니다.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "mlx-lm", "ollama", "strands-agents[ollama]", "numpy", "requests", "rdkit"], check=True)
print("✅ 패키지 준비 완료. 다음 셀로 넘어가세요.")

## 1. 설정
어느 팀을 증류할지 고릅니다. 내 팀으로 하려면 `TEAM_FILE`을 `"my_team"`으로 바꿉니다.

In [ ]:
import importlib, os, sys, time, warnings

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))

TEAM_FILE = "xray_team"      # TODO: 내 팀으로 하려면 "my_team"
module = importlib.import_module(TEAM_FILE)
team = module.TEAM

from agent_team import EXPERT_MODEL, STUDENT_MODEL
print(f"✅ {team.goal}")
print(f"   스승: {EXPERT_MODEL}   제자: {STUDENT_MODEL}")
print(f"   제자에게 줄 지시문: {team.instruction!r}")

## 2. 질문 만들고 스승에게 답 달게 하기
팀 파일의 질문 틀(`TRAIN_TEMPLATES`)에 낱말(`SUBJECTS`)을 넣어 질문을 만듭니다. `TEST_TEMPLATES`는 학습에 쓰지 않고, **처음 보는 유형**의 시험에만 씁니다.

끝나면 조합별 개수가 나옵니다. 한 조합이 대부분이면 제자는 그 답만 외웁니다. 그럴 때는 질문 틀을 고르게 늘립니다.

In [ ]:
train_rows, test_rows = team.build_distill_data(module.TRAIN_TEMPLATES, module.TEST_TEMPLATES, module.SUBJECTS)

import random
print("\n스승의 답 몇 개만 직접 보기 (스승이 틀리면 제자도 틀립니다):")
for r in random.Random(1).sample(train_rows, 8):
    print(f"   {r['experts']:<26} {r['question']}")

## 3. 학습 전 시험
제자가 스승과 **똑같은 전문가 조합**을 고르면 정답으로 칩니다.

In [ ]:
print("학습 전, 처음 보는 유형의 질문:")
before_test = team.exam("test", trained=False, show=6)
before_valid = team.exam("valid", trained=False)
print(f"\n📊 학습 전 정답률  본 유형 {before_valid:.0%}   처음 보는 유형 {before_test:.0%}")

## 4. 학습
설정은 600번 반복, 학습률 1e-4입니다(Step 2-6과 같음). 5분쯤 걸리고 그동안 아이맥의 팬이 돌 수 있습니다.

In [ ]:
team.train_student(iters=600, learning_rate=1e-4)

## 5. 학습 후 시험

In [ ]:
print("학습 후, 처음 보는 유형의 질문:")
after_test = team.exam("test", show=6)
after_valid = team.exam("valid")

print(f"\n📊 정답률 (스승과 같은 답을 고른 비율)")
print(f"                  학습 전    학습 후")
print(f"   본 유형          {before_valid:>5.0%}     {after_valid:>5.0%}")
print(f"   처음 보는 유형   {before_test:>5.0%}     {after_test:>5.0%}")

## 6. 제자를 지휘자 자리에 넣기
`team.ask()`의 `router` 자리에 증류한 모델을 넘깁니다. 전문가를 고르는 일만 작은 모델이 하고, 사진을 보고 분석하는 일은 큰 모델이 그대로 합니다.

In [ ]:
route_with_student = team.load_student()

# TODO: 질문을 바꿔 보세요! 학습 자료와 말투가 다른 질문일수록 스승과 다르게 고르는 일이 늘어납니다
questions = ["심장 그림자가 커 보이는지 봐 줘", "이런 사진 전에도 있었어?", "기흉 진료 지침의 근거를 찾아 줘"]
if TEAM_FILE != "xray_team":
    questions = ["이 수치면 괜찮은 거야?", "운동은 어떻게 하면 좋을까?", "전부 확인해서 정리해 줘"]
team.route_with_teacher(questions[0])      # 스승 모델을 미리 메모리에 올려 둡니다
for q in questions:
    t0 = time.time(); a = team.route_with_teacher(q); t1 = time.time(); b = route_with_student(q); t2 = time.time()
    print(f"{q}\n   스승 {t1 - t0:.2f}초  {a}\n   제자 {t2 - t1:.2f}초  {b}   {'⭕ 같음' if a == b else '❌ 다름'}\n")

In [ ]:
# 1분쯤 걸립니다
case = team.cases[0] if team.cases else ""
question = "이 사진과 비슷한 과거 사례가 있어?" if TEAM_FILE == "xray_team" else "검사 수치 가운데 이상한 게 있어?"
result = team.ask(question, case=case, router=route_with_student)

## 7. 서비스에서도 쓰기
학습된 Adapter는 `distill/<팀 이름>/adapter`에 저장됩니다. Step 2-3의 서비스를 **다시 띄우면** 자동으로 찾아 씁니다.

```bash
.venv/bin/python team_service.py xray_team
```
웹 화면의 "전문가를 고르는 모델"에 **증류한 작은 모델**이 생깁니다. 큰 모델로 바꿔 가며 고르는 시간을 비교해 보세요.

## 정리
- 주제가 신약에서 엑스레이로 바뀌어도 증류의 순서는 같습니다: 질문 모으기 → 스승이 답 달기 → 학습 → 시험
- 증류한 것은 **라우팅뿐**입니다. 사진을 보는 일을 작은 모델에 옮기려면 사진을 다루는 학습 도구와, 무엇보다 **믿을 수 있는 정답**이 필요합니다. 스승이 사진을 틀리게 읽으면 제자는 그 틀린 답을 배웁니다
- 처음 보는 유형의 정답률이 낮으면 질문 틀이 부족한 것입니다. 실제 서비스에서는 사용자가 실제로 한 질문 기록을 씁니다
- 틀린 선택의 비용을 생각하세요. 이 팀은 알 수 없는 답이 나오면 전문가를 전부 부릅니다. 느려지더라도 빠뜨리지 않는 쪽입니다

## 더 해 보기
- `TEAM_FILE = "my_team"`으로 바꿔 내 팀을 증류해 보세요. 질문 틀이 적으면 정답률이 어떻게 되나요?
- 질문 틀을 10개 더 넣고 다시 학습하면 처음 보는 유형의 정답률이 오르나요?